# Push Cleaned Dataset to Hugging Face Hub

This notebook:
1. Loads the cleaned dataset
2. Renames columns to `code` and `feedback`
3. Pushes to Hugging Face Hub

**Requirements**: `huggingface_hub` and authentication token

## 1. Setup and Imports

In [1]:
import pandas as pd
import json
from pathlib import Path
from datasets import Dataset, DatasetDict
from huggingface_hub import login, HfApi
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully!")

✓ Libraries loaded successfully!


## 2. Hugging Face Authentication

You need to authenticate with your Hugging Face token. You can:
- Run `huggingface-cli login` in terminal, or
- Uncomment and use the `login()` function below

In [2]:
# Option 1: Login interactively (will prompt for token)
# login()

# Option 2: Login with token directly
# login(token="your_token_here")

# Check if logged in
try:
    api = HfApi()
    user_info = api.whoami()
    print(f"✓ Logged in as: {user_info['name']}")
except Exception as e:
    print("✗ Not logged in. Please run: huggingface-cli login")
    print(f"  Error: {e}")

✓ Logged in as: matis35


## 3. Load Cleaned Dataset

In [8]:
print("="*100)
print("LOADING CLEANED DATASET")
print("="*100)
print()

# Load from CSV
dataset_path = Path('../data/cleaned_dataset_no_cot.csv')

if not dataset_path.exists():
    print(f"✗ Dataset not found at: {dataset_path}")
    print("  Please run the data_cleaning.ipynb notebook first!")
else:
    df = pd.read_csv(dataset_path)
    print(f"✓ Loaded {len(df):,} samples from cleaned_dataset_no_cot.csv")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"Shape: {df.shape}")

LOADING CLEANED DATASET

✓ Loaded 11,806 samples from cleaned_dataset_no_cot.csv

Columns: ['code_id', 'author_id', 'code_snippet', 'generated_feedback']
Shape: (11806, 4)


## 4. Prepare Dataset for Hub

In [9]:
print("="*100)
print("PREPARING DATASET FOR HUB")
print("="*100)
print()

# Create a new dataframe with only code and feedback columns
df_hub = pd.DataFrame({
    'code': df['code_snippet'],
    'feedback': df['generated_feedback']
})

print(f"✓ Created dataset with columns: {df_hub.columns.tolist()}")
print(f"  Total samples: {len(df_hub):,}")
print()

# Show statistics
print("Dataset statistics:")
print(f"  Average code length: {df_hub['code'].str.len().mean():.0f} characters")
print(f"  Average feedback length: {df_hub['feedback'].str.len().mean():.0f} characters")
print(f"  Unique feedback messages: {df_hub['feedback'].nunique():,}")
print()

# Check for nulls
nulls = df_hub.isnull().sum()
if nulls.sum() > 0:
    print("⚠ Warning: Found null values:")
    print(nulls[nulls > 0])
else:
    print("✓ No null values found")

PREPARING DATASET FOR HUB

✓ Created dataset with columns: ['code', 'feedback']
  Total samples: 11,806

Dataset statistics:
  Average code length: 289 characters
  Average feedback length: 171 characters
  Unique feedback messages: 11,061

✓ No null values found


## 5. Preview Dataset

In [10]:
print("="*100)
print("DATASET PREVIEW (3 Random Samples)")
print("="*100)
print()

samples = df_hub.sample(3)

for idx, (i, row) in enumerate(samples.iterrows(), 1):
    print(f"Sample {idx}:")
    print("-" * 100)
    print(f"\nCode:")
    print(row['code'][:300] + ("..." if len(row['code']) > 300 else ""))
    print(f"\nFeedback:")
    print(row['feedback'][:200] + ("..." if len(row['feedback']) > 200 else ""))
    print("\n" + "="*100 + "\n")

DATASET PREVIEW (3 Random Samples)

Sample 1:
----------------------------------------------------------------------------------------------------

Code:
char *my_strstr(char *str, char const *to_find)
{
    if (str[0] == '\0') {
        return 0;
    }
    for (int i = 0; to_find[i] != '\0'; i++) {
        if (str[i] != to_find[i]) {
            return (my_strstr(str + 1, to_find));
        }
    }
    return (str);
}

Feedback:
When iterating through the string, you are only checking for a match starting from the first character of the string to search within. This approach will miss potential matches that start at later pos...


Sample 2:
----------------------------------------------------------------------------------------------------

Code:
#include <stddef.h>

static int my_strlen(char const *str)
{
    int i = 0;

    for (; str[i] != '\0'; i++);
    return i;
}

static int my_strncmp(char const *s1, char const *s2, int n)
{
    int sum1 = 0;
    int sum2 = 0;

    if (s1 == N

## 6. Convert to Hugging Face Dataset

In [11]:
print("="*100)
print("CONVERTING TO HUGGING FACE DATASET")
print("="*100)
print()

# Create Hugging Face Dataset
dataset = Dataset.from_pandas(df_hub, preserve_index=False)

print(f"✓ Created Hugging Face Dataset")
print(f"  Features: {dataset.features}")
print(f"  Number of rows: {len(dataset):,}")
print()

# Show first example
print("First example:")
print(dataset[0])

CONVERTING TO HUGGING FACE DATASET

✓ Created Hugging Face Dataset
  Features: {'code': Value('string'), 'feedback': Value('string')}
  Number of rows: 11,806

First example:
{'code': 'int my_compute_power_rec(int nb, int p)\n{\n    if (p == 0) {\n        return (1);\n    }\n    if (p < 0) {\n        return (0);\n    }\n    return (nb * (my_compute_power_rec(nb, p - 1)));\n}', 'feedback': 'Consider the behavior of the function when the first parameter is zero.'}


## 7. Create Train/Validation/Test Split

Creates three splits:
- **Train**: 80% - For training the model
- **Validation**: 10% - For hyperparameter tuning and model selection
- **Test**: 10% - For final evaluation

In [13]:
print("="*100)
print("CREATING TRAIN/VALIDATION/TEST SPLIT")
print("="*100)
print()

# First split: separate test set (10%)
train_val_test = dataset.train_test_split(test_size=0.1, seed=42)

# Second split: separate validation from train (10% of remaining = ~11% of total)
# This gives us approximately 80% train, 10% validation, 10% test
train_val = train_val_test['train'].train_test_split(test_size=0.111, seed=42)

# Create the final dataset dictionary
dataset_split = DatasetDict({
    'train': train_val['train'],
    'validation': train_val['test'],
    'test': train_val_test['test']
})

print(f"✓ Created train/validation/test split:")
print(f"  Train:      {len(dataset_split['train']):,} samples ({len(dataset_split['train'])/len(dataset)*100:.1f}%)")
print(f"  Validation: {len(dataset_split['validation']):,} samples ({len(dataset_split['validation'])/len(dataset)*100:.1f}%)")
print(f"  Test:       {len(dataset_split['test']):,} samples ({len(dataset_split['test'])/len(dataset)*100:.1f}%)")
print(f"  Total:      {len(dataset):,} samples")
print()

# Use the split dataset for pushing
dataset_to_push = dataset_split

# Uncomment the line below if you want to push the full dataset without splits
# dataset_to_push = dataset

CREATING TRAIN/VALIDATION/TEST SPLIT

✓ Created train/validation/test split:
  Train:      9,445 samples (80.0%)
  Validation: 1,180 samples (10.0%)
  Test:       1,181 samples (10.0%)
  Total:      11,806 samples



## 8. Configure Dataset Information

In [18]:
# Configure your dataset details
DATASET_NAME = "matis35/RAFT"  # Change this!
DATASET_DESCRIPTION = """# Code Feedback Dataset - Retrieval augmented feedback training

This dataset contains C code snippets with generated feedback for errors and improvements.

## Dataset Description

- **Total samples**: {num_samples}
- **Language**: C
- **Task**: Code error detection and feedback generation

## Dataset Structure

Each example contains:
- `code`: C code snippet
- `feedback`: Generated feedback describing errors or improvements

## Cleaning Process

The dataset has been cleaned to remove:
1. Empty code snippets
2. Test files (containing Criterion test framework)
3. Entries with "No functional error detected" feedback

## Usage

```python
from datasets import load_dataset

dataset = load_dataset("{dataset_name}")
```

## License

Please specify your license here.
""".format(num_samples=len(dataset_to_push), dataset_name=DATASET_NAME)

print("="*100)
print("DATASET CONFIGURATION")
print("="*100)
print()
print(f"Dataset name: {DATASET_NAME}")
print()
print("⚠ IMPORTANT: Update DATASET_NAME above with your username!")
print("   Example: 'username/code-feedback-dataset'")
print()

DATASET CONFIGURATION

Dataset name: matis35/RAFT

⚠ IMPORTANT: Update DATASET_NAME above with your username!
   Example: 'username/code-feedback-dataset'



## 9. Push to Hugging Face Hub

**IMPORTANT**: Make sure to update `DATASET_NAME` in the cell above before running this!

In [20]:
# Safety check
if DATASET_NAME != "matis35/RAFT":
    print("✗ ERROR: Please update DATASET_NAME in the configuration cell above!")
    print("  Change 'your-username' to your actual Hugging Face username.")
else:
    print("="*100)
    print("PUSHING TO HUGGING FACE HUB")
    print("="*100)
    print()
    print(f"Pushing dataset to: {DATASET_NAME}")
    print("This may take a few minutes...")
    print()
    
    try:
        # Push to hub
        dataset_to_push.push_to_hub(
            DATASET_NAME,
            private=False,  # Set to True if you want a private dataset
            commit_message="Initial upload of cleaned code-feedback dataset"
        )
        
        print("\n" + "="*100)
        print("✓ SUCCESS!")
        print("="*100)
        print()
        print(f"Dataset successfully pushed to: https://huggingface.co/datasets/{DATASET_NAME}")
        print()
        print("You can now:")
        print(f"  1. View your dataset: https://huggingface.co/datasets/{DATASET_NAME}")
        print(f"  2. Load it with: load_dataset('{DATASET_NAME}')")
        print(f"  3. Edit the dataset card to add more information")
        
    except Exception as e:
        print("\n" + "="*100)
        print("✗ ERROR DURING UPLOAD")
        print("="*100)
        print()
        print(f"Error: {e}")
        print()
        print("Common issues:")
        print("  1. Not logged in - Run: huggingface-cli login")
        print("  2. Invalid dataset name format")
        print("  3. No write permissions")
        print("  4. Network issues")

PUSHING TO HUGGING FACE HUB

Pushing dataset to: matis35/RAFT
This may take a few minutes...



Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✓ SUCCESS!

Dataset successfully pushed to: https://huggingface.co/datasets/matis35/RAFT

You can now:
  1. View your dataset: https://huggingface.co/datasets/matis35/RAFT
  2. Load it with: load_dataset('matis35/RAFT')
  3. Edit the dataset card to add more information


## 10. Create and Upload Dataset Card (Optional)

Create a detailed README for your dataset

In [21]:
# This will create a README.md file that you can upload to your dataset
readme_content = f"""---
language:
- en
task_categories:
- text-generation
- text2text-generation
tags:
- code
- feedback
- error-detection
- c-programming
size_categories:
- 1K<n<10K
---

{DATASET_DESCRIPTION}

## Dataset Statistics

- **Total examples**: {len(dataset_to_push):,}
- **Average code length**: {df_hub['code'].str.len().mean():.0f} characters
- **Average feedback length**: {df_hub['feedback'].str.len().mean():.0f} characters
- **Unique feedback messages**: {df_hub['feedback'].nunique():,}

## Example

```python
{{
  "code": "{dataset_to_push[0]['code'][:200].replace('\n', '\\n')}...",
  "feedback": "{dataset_to_push[0]['feedback'][:200].replace('\n', '\\n')}..."
}}
```

## Citation

If you use this dataset, please cite:

```
@dataset{{code_feedback_dataset,
  title={{Code Feedback Dataset}},
  author={{Matis Codjia}},
  year={{2025}},
  publisher={{Hugging Face}},
  url={{https://huggingface.co/datasets/{DATASET_NAME}}}
}}
```
"""

# Save README locally
readme_path = Path('../data/DATASET_README.md')
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"✓ Dataset README saved to: {readme_path}")
print("\nYou can copy this content to your dataset's README on Hugging Face Hub")

SyntaxError: f-string expression part cannot include a backslash (1123414827.py, line 48)

## 11. Verify Upload

In [23]:
# Verify by loading the dataset from the hub
if DATASET_NAME == "matis35/RAFT":
    print("="*100)
    print("VERIFYING UPLOAD")
    print("="*100)
    print()
    
    try:
        from datasets import load_dataset
        
        print(f"Loading dataset from hub: {DATASET_NAME}")
        loaded_dataset = load_dataset(DATASET_NAME)
        
        print("\n✓ Dataset loaded successfully!")
        print(f"\nDataset info:")
        print(loaded_dataset)
        
        print("\nFirst example from loaded dataset:")
        if isinstance(loaded_dataset, DatasetDict):
            print(loaded_dataset['train'][0])
        else:
            print(loaded_dataset[0])
            
    except Exception as e:
        print(f"\n⚠ Could not verify upload yet: {e}")
        print("   The dataset might still be processing. Try again in a few minutes.")
else:
    print("⚠ Update DATASET_NAME first to verify upload")

VERIFYING UPLOAD

Loading dataset from hub: matis35/RAFT


README.md:   0%|          | 0.00/525 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/203k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/205k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9445 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1180 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1181 [00:00<?, ? examples/s]


✓ Dataset loaded successfully!

Dataset info:
DatasetDict({
    train: Dataset({
        features: ['code', 'feedback'],
        num_rows: 9445
    })
    validation: Dataset({
        features: ['code', 'feedback'],
        num_rows: 1180
    })
    test: Dataset({
        features: ['code', 'feedback'],
        num_rows: 1181
    })
})

First example from loaded dataset:
{'code': 'int my_strlen(char const *str);\n\nint my_strcmp(char const *s1, char const *s2)\n{\n    int i = 0;\n    int l1 = my_strlen(s1);\n    int l2 = my_strlen(s2);\n\n    if (l1 > l2) {\n        return 1;\n    }\n    if (l1 < l2) {\n        return -1;\n    }\n    while (i < l1) {\n        if (s1[i] > s2[i]) {\n            return 1;\n        }\n        if (s1[i] < s2[i]) {\n            return -1;\n        }\n        i++;\n    }\n    return 0;\n}', 'feedback': 'When comparing strings, the function should first check for differences in characters before comparing lengths.\n\nThe student can identify his bug by look

## Summary

This notebook has:
1. ✓ Loaded the cleaned dataset
2. ✓ Renamed columns to `code` and `feedback`
3. ✓ Converted to Hugging Face Dataset format
4. ✓ Pushed to Hugging Face Hub
5. ✓ Created a dataset README

Your dataset is now available at: `https://huggingface.co/datasets/{DATASET_NAME}`